# 03. SVM Hyperparameter Tuning
## GridSearchCV with Stratified Group K-Fold Cross-Validation

This notebook tunes the Support Vector Machine hyperparameters over kernels and regularization strengths without touching the holdout test set.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")
from src.features.preprocessing import load_dataset, get_X_y_groups
from src.data.splitting import subject_level_train_test_split
from src.models.tune_svm import tune_svm
from src.evaluation.metrics import compute_metrics
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

%matplotlib inline
sns.set_theme(style="whitegrid")


### 1. Load Data and Train/Test Split


In [ ]:
df = load_dataset()
df_train, df_test = subject_level_train_test_split(df)
X_train, y_train, groups_train = get_X_y_groups(df_train)
X_test, y_test, groups_test = get_X_y_groups(df_test)


### 2. Execute Hyperparameter Search


In [ ]:
best_pipeline, best_params, df_cv_results = tune_svm(df_train)
print("\nBest Hyperparameters Found:", best_params)
df_cv_results.head(10)


### 3. Evaluate Tuned Model on Untouched Test Split


In [ ]:
y_pred = best_pipeline.predict(X_test)
final_metrics = compute_metrics(y_test, y_pred, label="Tuned_SVM")
for k, v in final_metrics.items():
    print(f"{k}: {v}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


### 4. Confusion Matrix Display


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax1, cmap="Blues")
ax1.set_title("Confusion Matrix (Counts)")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax2, cmap="Blues", normalize="true")
ax2.set_title("Confusion Matrix (Normalized)")
plt.tight_layout()
plt.show()
